In [1]:
!pip install kagglehub

In [2]:
import pandas as pd
import numpy as np
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmedabbas757/coffee-sales")

print("Path to dataset files:", path)
files = os.listdir(path)
print("Files in directory:", files)

100%|█████████████████████████████████████████████████████████████████████████████| 8.23M/8.23M [00:02<00:00, 4.29MB/s]

Extracting files...
Path to dataset files: C:\Users\shado\.cache\kagglehub\datasets\ahmedabbas757\coffee-sales\versions\1
Files in directory: ['Coffee Shop Sales.xlsx']


In [4]:
excel_file_path = os.path.join(path, 'Coffee Shop Sales.xlsx')
df = pd.read_excel(excel_file_path)

df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [5]:
print(df.shape)

df.info()

print(df.isnull().sum())

(149116, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    149116 non-null  int64         
 1   transaction_date  149116 non-null  datetime64[ns]
 2   transaction_time  149116 non-null  object        
 3   transaction_qty   149116 non-null  int64         
 4   store_id          149116 non-null  int64         
 5   store_location    149116 non-null  object        
 6   product_id        149116 non-null  int64         
 7   unit_price        149116 non-null  float64       
 8   product_category  149116 non-null  object        
 9   product_type      149116 non-null  object        
 10  product_detail    149116 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(5)
memory usage: 12.5+ MB
transaction_id      0
transaction_date    0
transaction_time    0
transaction_q

##  Data Sabotage (Simulating Real-World Data Issues)
> **Note:** To demonstrate data cleaning skills, we are intentionally injecting realistic errors (duplicates, missing values, outliers, and text inconsistencies) into our clean dataset, which we will then clean in the next phase.

In [6]:
import pandas as pd
import numpy as np
import os

dirty_df = df.copy()

np.random.seed(42)

duplicates = dirty_df.sample(n=2000)
dirty_df = pd.concat([dirty_df, duplicates], ignore_index=True)

dirty_df.loc[dirty_df.sample(frac=0.03).index, 'transaction_qty'] = np.nan
dirty_df.loc[dirty_df.sample(frac=0.02).index, 'unit_price'] = np.nan
dirty_df.loc[dirty_df.sample(frac=0.02).index, 'product_category'] = np.nan

def mess_up_text(text):
    if pd.isna(text):
        return text
    choice = np.random.choice([1, 2, 3, 4])
    if choice == 1:
        return f"  {text}  " 
    elif choice == 2:
        return text.lower()  
    elif choice == 3:
        return text.upper()  
    return text

dirty_df['product_category'] = dirty_df['product_category'].apply(mess_up_text)
dirty_df['store_location'] = dirty_df['store_location'].apply(mess_up_text)

dirty_df.loc[dirty_df.sample(n=50).index, 'transaction_qty'] = -5
dirty_df.loc[dirty_df.sample(n=30).index, 'unit_price'] = 999.0

dirty_df.to_excel("Dirty_Coffee_Shop_Sales.xlsx", index=False)
print("Dirty dataset 'Dirty_Coffee_Shop_Sales.xlsx' successfully created with realistic errors!")

Dirty dataset 'Dirty_Coffee_Shop_Sales.xlsx' successfully created with realistic errors!


In [7]:
dirty_df_raw = pd.read_excel("Dirty_Coffee_Shop_Sales.xlsx")

print("Dirty Dataset Shape (Rows, Columns):")
print(dirty_df_raw.shape)

Dirty Dataset Shape (Rows, Columns):
(151116, 11)


In [8]:
print("Missing Values in Dirty Dataset:")
print(dirty_df_raw.isnull().sum())
print("-" * 50)

duplicate_count = dirty_df_raw.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

Missing Values in Dirty Dataset:
transaction_id         0
transaction_date       0
transaction_time       0
transaction_qty     4532
store_id               0
store_location         0
product_id             0
unit_price          3022
product_category    3022
product_type           0
product_detail         0
dtype: int64
--------------------------------------------------
Number of duplicate rows found: 123


In [9]:
invalid_qty = dirty_df_raw[dirty_df_raw['transaction_qty'] < 0]
extreme_price = dirty_df_raw[dirty_df_raw['unit_price'] > 500]

print(f"Number of rows with negative quantity: {len(invalid_qty)}")
print(f"Number of rows with extreme price ($999): {len(extreme_price)}")

Number of rows with negative quantity: 50
Number of rows with extreme price ($999): 30


In [10]:
print("Inconsistent Product Categories:")
print(dirty_df_raw['product_category'].unique()[:10])

Inconsistent Product Categories:
['coffee' '  Tea  ' '  Drinking Chocolate  ' '  Coffee  ' '  Bakery  '
 'COFFEE' 'Tea' 'Coffee' 'TEA' nan]


### Data Cleaning

In [12]:

cleaned_df = dirty_df_raw.copy()

for col in ['product_category', 'store_location']:
    cleaned_df[col] = cleaned_df[col].astype(str).str.strip().str.title()
    cleaned_df[col] = cleaned_df[col].replace('Nan', np.nan)

cleaned_df.loc[cleaned_df['unit_price'] > 500, 'unit_price'] = np.nan

cleaned_df['unit_price'] = cleaned_df.groupby('product_id')['unit_price'].transform(lambda x: x.fillna(x.median()))

cleaned_df['transaction_qty'] = cleaned_df['transaction_qty'].abs()
cleaned_df['transaction_qty'] = cleaned_df['transaction_qty'].fillna(1).astype(int)

category_map = cleaned_df.dropna(subset=['product_category']).set_index('product_type')['product_category'].to_dict()
cleaned_df['product_category'] = cleaned_df['product_category'].fillna(cleaned_df['product_type'].map(category_map))

cleaned_df.drop_duplicates(inplace=True)

print("--- Verification After Cleaning ---")
print("Remaining Missing Values (Nulls) per column:")
print(cleaned_df.isnull().sum())
print("-" * 50)
print(f"Remaining duplicate rows count: {cleaned_df.duplicated().sum()}")
print("-" * 50)
print("Standardized Product Categories:")
print(cleaned_df['product_category'].unique())

--- Verification After Cleaning ---
Remaining Missing Values (Nulls) per column:
transaction_id      0
transaction_date    0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64
--------------------------------------------------
Remaining duplicate rows count: 0
--------------------------------------------------
Standardized Product Categories:
['Coffee' 'Tea' 'Drinking Chocolate' 'Bakery' 'Flavours' 'Loose Tea'
 'Coffee Beans' 'Packaged Chocolate' 'Branded']


## Financial Calculations & Customer Sentiment Simulation
> **Goal:** In this phase, we dynamically compute financial metrics (Unit Cost, Revenue, Cost, and Profit) based on industry-standard profit margins. Additionally, we simulate customer reviews and ratings (sampling 10% of the transactions) with built-in logic to mimic real-world operational scenarios, such as morning rush hours and price-sensitive feedback.

In [14]:
margin_map = {
    'Coffee': 0.75,
    'Tea': 0.70,
    'Drinking Chocolate': 0.65,
    'Bakery': 0.45,
    'Flavours': 0.80,
    'Loose Tea': 0.60,
    'Coffee Beans': 0.50,
    'Branded': 0.35,
    'Packaged Chocolate': 0.40
}

cleaned_df['margin_rate'] = cleaned_df['product_category'].map(margin_map).fillna(0.50)
cleaned_df['unit_cost'] = round(cleaned_df['unit_price'] * (1 - cleaned_df['margin_rate']), 2)
cleaned_df['total_revenue'] = cleaned_df['transaction_qty'] * cleaned_df['unit_price']
cleaned_df['total_cost'] = cleaned_df['transaction_qty'] * cleaned_df['unit_cost']
cleaned_df['total_profit'] = cleaned_df['total_revenue'] - cleaned_df['total_cost']
cleaned_df.drop(columns=['margin_rate'], inplace=True)

np.random.seed(42)
review_sample = cleaned_df.sample(frac=0.10).copy()

def generate_review(row):
    time_str = str(row['transaction_time'])
    hour = int(time_str.split(':')[0]) if ':' in time_str else 12
    
    if (7 <= hour <= 9) and (row['store_location'] == 'Lower Manhattan'):
        rating = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2])
        review_text = np.random.choice([
            "Long waiting time in the morning rush!",
            "Service was extremely slow today.",
            "The queue was out the door. Coffee was okay but not worth the wait.",
            "Understaffed morning shift."
        ])
    elif row['unit_price'] > 15.0:
        rating = np.random.choice([2, 3, 4], p=[0.3, 0.5, 0.2])
        review_text = np.random.choice([
            "Overpriced for the quantity.",
            "Good product but too expensive.",
            "Nice packaging, but pricing is too high.",
            "The taste is fine, but price needs to be adjusted."
        ])
    else:
        rating = np.random.choice([4, 5], p=[0.3, 0.7])
        review_text = np.random.choice([
            "Amazing coffee and great atmosphere!",
            "Fast service, highly recommend.",
            "My favorite morning spot!",
            "Friendly staff and great quality.",
            "Perfect brew every single time!"
        ])
        
    return pd.Series([rating, review_text])

review_sample[['rating', 'review_text']] = review_sample.apply(generate_review, axis=1)
reviews_df = review_sample[['transaction_id', 'rating', 'review_text']].copy()

print("Columns added to Sales DataFrame: unit_cost, total_revenue, total_cost, total_profit")
print(f"Generated Reviews DataFrame with {len(reviews_df)} records.")

Columns added to Sales DataFrame: unit_cost, total_revenue, total_cost, total_profit
Generated Reviews DataFrame with 14916 records.


In [15]:
cleaned_df.to_csv("Cleaned_Coffee_Shop_Sales.csv", index=False)
reviews_df.to_csv("Simulated_Customer_Reviews.csv", index=False)

print("Files saved successfully")

Files saved successfully
